# ESMC Sparse Autoencoders (SAE) - Interpretable Feature Extraction

Extract and visualize interpretable biological features from ESMC using Sparse Autoencoders.

**Key Features:**
- Extract 16,384 interpretable sparse features from ESMC embeddings
- Each residue represents ~64 active features
- Features correspond to: binding sites, motifs, biophysical properties, evolutionary signals
- Analyze feature activations across sequences
- Visualize feature patterns and their biological meaning


In [ ]:
!pip install -q torch transformers accelerate einops biotite biopython scikit-learn pandas tqdm ipywidgets

In [ ]:
# Install ESM from GitHub
!pip install -q git+https://github.com/Biohub/esm.git@main

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import json
from tqdm import tqdm
from IPython.display import display, Markdown, HTML

# ESM imports
from esm.models import ESMCForCausalLM
from esm.tokenization import TokenizerBase
from esm.models.sae import SAE  # Sparse Autoencoder

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
# Model selection
ESMC_MODEL = "esmc_600m"
SAE_VARIANT = "default"  # Options: "default", "layer-specific"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# SAE parameters
NUM_FEATURES = 16384  # Total sparse features
FEATURES_PER_RESIDUE = 64  # Typical sparsity

# Data
OUTPUT_DIR = "/content/esmc_sae_output"
Path(OUTPUT_DIR).mkdir(exist_ok=True)

print(f"ESMC Model: {ESMC_MODEL}")
print(f"SAE Features: {NUM_FEATURES}")
print(f"Sparsity: ~{FEATURES_PER_RESIDUE} active features per residue")
print(f"Device: {DEVICE}")
print(f"Output directory: {OUTPUT_DIR}")

## Load ESMC and SAE Models

In [ ]:
print("Loading ESMC model...")
esmc = ESMCForCausalLM.from_pretrained(f"esm2/esmc_{ESMC_MODEL}").to(DEVICE)
esmc.eval()

print("Loading SAE...")
sae = SAE.from_pretrained(f"esm2/esmc_sae_{SAE_VARIANT}").to(DEVICE)
sae.eval()

tokenizer = TokenizerBase()

print(f"✓ ESMC loaded: {sum(p.numel() for p in esmc.parameters())/1e6:.1f}M parameters")
print(f"✓ SAE loaded with {NUM_FEATURES} features")

## Utility Functions

In [ ]:
def load_fasta(filepath: str) -> Dict[str, str]:
    """Load FASTA file into dictionary."""
    sequences = {}
    with open(filepath) as f:
        current_id = None
        current_seq = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if current_id is not None:
                    sequences[current_id] = "".join(current_seq)
                current_id = line[1:].split()[0]
                current_seq = []
            else:
                current_seq.append(line)
        if current_id is not None:
            sequences[current_id] = "".join(current_seq)
    return sequences

def extract_sae_features(hidden_states: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """Extract SAE features from ESMC hidden states.
    
    Returns:
        features: [seq_len, num_features] - Dense feature matrix
        feature_mask: [seq_len, num_features] - Binary mask of active features
    """
    with torch.no_grad():
        # SAE encoding (sparse)
        encoded = sae.encode(hidden_states)
        
        # Get active features (non-zero)
        feature_mask = (encoded != 0).float()
        
        return encoded, feature_mask

def get_top_features(feature_matrix: np.ndarray, top_k: int = 20) -> List[Tuple[int, float]]:
    """Get top activated features by mean activation across sequence."""
    mean_activation = np.mean(np.abs(feature_matrix), axis=0)
    top_indices = np.argsort(mean_activation)[::-1][:top_k]
    return [(int(idx), float(mean_activation[idx])) for idx in top_indices]

## Load Sequences

In [ ]:
#@markdown Upload FASTA file or use example sequences

from google.colab import files

input_mode = "paste"  # "upload" or "paste"

if input_mode == "upload":
    print("Upload your FASTA file:")
    uploaded = files.upload()
    fasta_file = list(uploaded.keys())[0]
    sequences = load_fasta(fasta_file)
else:
    # Example: ATP synthase subunit
    sequences = {
        "ATP_synthase_example": "MKTIIALSYIFCLVFADYKDDDKGWYLQKPAILFEEFLTPVIKEAKKTDYFPHFDLSHGSAQVQGHEVQAILVVDGIDLHDNLRPQSTRKDDHWLQLMQFGVDPLPPAVRESVPSLL"
    }

print(f"Loaded {len(sequences)} sequences")
for seq_id, seq in list(sequences.items())[:3]:
    print(f"  {seq_id}: {len(seq)} aa")

## Extract SAE Features

In [ ]:
sae_features = {}
feature_masks = {}
top_features_dict = {}

with torch.no_grad():
    for seq_id, sequence in tqdm(sequences.items(), desc="Extracting SAE features"):
        # Tokenize
        tokens = tokenizer.tokenize(sequence)
        tokens = torch.tensor([tokens]).to(DEVICE)
        
        # Get ESMC embeddings
        output = esmc(tokens, output_hidden_states=True)
        hidden_states = output.hidden_states[-1]  # Last layer
        
        # Extract SAE features
        features, mask = extract_sae_features(hidden_states[0, 1:-1, :])  # Remove special tokens
        
        sae_features[seq_id] = features.cpu().numpy()
        feature_masks[seq_id] = mask.cpu().numpy()
        
        # Get top features
        top_features_dict[seq_id] = get_top_features(sae_features[seq_id], top_k=20)

print(f"\n✓ Extracted SAE features for {len(sae_features)} sequences")
for seq_id in list(sae_features.keys())[:1]:
    print(f"  {seq_id}: shape {sae_features[seq_id].shape}")
    print(f"    Sparsity: {np.mean(np.sum(feature_masks[seq_id] > 0, axis=1)):.1f} active features per residue")

## Feature Analysis

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

for seq_id in sequences.keys():
    features = sae_features[seq_id]
    mask = feature_masks[seq_id]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"{seq_id} - SAE Feature Analysis", fontsize=14, fontweight='bold')
    
    # 1. Feature activation heatmap (top 100 features)
    top_100_features = np.argsort(np.mean(np.abs(features), axis=0))[::-1][:100]
    ax = axes[0, 0]
    im = ax.imshow(features[:, top_100_features].T, aspect='auto', cmap='RdBu_r', interpolation='nearest')
    ax.set_xlabel('Position')
    ax.set_ylabel('Feature Index')
    ax.set_title('Top 100 Active Features Heatmap')
    plt.colorbar(im, ax=ax, label='Activation')
    
    # 2. Number of active features per position
    ax = axes[0, 1]
    active_per_pos = np.sum(mask > 0, axis=1)
    ax.plot(active_per_pos, linewidth=2, color='steelblue')
    ax.fill_between(range(len(active_per_pos)), active_per_pos, alpha=0.3, color='steelblue')
    ax.set_xlabel('Position')
    ax.set_ylabel('Number of Active Features')
    ax.set_title('Feature Sparsity per Position')
    ax.grid(True, alpha=0.3)
    
    # 3. Top feature activations
    ax = axes[1, 0]
    top_feats = top_features_dict[seq_id][:15]
    feat_ids = [f[0] for f in top_feats]
    feat_vals = [f[1] for f in top_feats]
    ax.barh(range(len(feat_ids)), feat_vals, color='coral')
    ax.set_yticks(range(len(feat_ids)))
    ax.set_yticklabels([f'Feature {fid}' for fid in feat_ids], fontsize=9)
    ax.set_xlabel('Mean Activation')
    ax.set_title('Top 15 Most Active Features')
    ax.invert_yaxis()
    
    # 4. Feature activation distribution
    ax = axes[1, 1]
    mean_activations = np.mean(np.abs(features), axis=0)
    ax.hist(mean_activations[mean_activations > 0], bins=50, color='mediumseagreen', edgecolor='black')
    ax.set_xlabel('Mean Activation')
    ax.set_ylabel('Number of Features')
    ax.set_title('Distribution of Feature Activations')
    ax.set_yscale('log')
    
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{seq_id}_sae_analysis.png", dpi=150, bbox_inches='tight')
    plt.show()

## Feature Interpretation

In [ ]:
# Prepare feature reports
for seq_id in sequences.keys():
    features = sae_features[seq_id]
    mask = feature_masks[seq_id]
    
    report = []
    report.append(f"# {seq_id} - SAE Feature Report\n")
    report.append(f"**Sequence length:** {features.shape[0]} residues\n")
    report.append(f"**Total features:** {NUM_FEATURES}\n")
    report.append(f"**Average active features/residue:** {np.mean(np.sum(mask > 0, axis=1)):.1f}\n")
    report.append(f"**Sparsity ratio:** {100 * np.sum(mask > 0) / (features.shape[0] * NUM_FEATURES):.2f}%\n\n")
    
    report.append("## Top 20 Most Active Features\n")
    report.append("| Feature ID | Mean Activation |\n")
    report.append("|---|---|\n")
    for feat_id, activation in top_features_dict[seq_id]:
        report.append(f"| {feat_id} | {activation:.4f} |\n")
    
    report_text = "".join(report)
    
    # Save report
    with open(f"{OUTPUT_DIR}/{seq_id}_sae_report.md", "w") as f:
        f.write(report_text)
    
    display(Markdown(report_text))

## Save Results

In [ ]:
# Save SAE features
for seq_id in sae_features.keys():
    # Full feature matrix
    np.save(f"{OUTPUT_DIR}/{seq_id}_sae_features.npy", sae_features[seq_id])
    
    # Feature mask (sparsity pattern)
    np.save(f"{OUTPUT_DIR}/{seq_id}_feature_mask.npy", feature_masks[seq_id])
    
    # Top features as CSV
    top_df = pd.DataFrame(
        top_features_dict[seq_id],
        columns=['feature_id', 'mean_activation']
    )
    top_df.to_csv(f"{OUTPUT_DIR}/{seq_id}_top_features.csv", index=False)

print(f"✓ All results saved to {OUTPUT_DIR}/")
print(f"  - SAE features: {{seq_id}}_sae_features.npy")
print(f"  - Feature mask: {{seq_id}}_feature_mask.npy")
print(f"  - Top features: {{seq_id}}_top_features.csv")
print(f"  - Analysis plots: {{seq_id}}_sae_analysis.png")
print(f"  - Feature reports: {{seq_id}}_sae_report.md")